In [6]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import random
from collections import deque
import heapq

# ==========================================
# 1. CORE DUNGEON LOGIC (Connected from Parts)
# ==========================================
def manhattan(cell1, cell2): return abs(cell1[0] - cell2[0]) + abs(cell1[1] - cell2[1])

def get_neighbors(grid, row, col):
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    return [(row + dr, col + dc) for dr, dc in directions 
            if 0 <= row + dr < len(grid) and 0 <= col + dc < len(grid[0]) and grid[row + dr][col + dc] == 1]

def reconstruct_path(parent, start, goal):
    path, current = [], goal
    while current != start:
        path.append(current)
        current = parent[current]
    path.append(start)
    return path[::-1]

def generate_dungeon(rows=20, cols=24, start=(1,1), goal=(18,22)):
    grid = [[0 for _ in range(cols)] for _ in range(rows)]
    r, c = start
    grid[r][c] = 1
    while (r, c) != goal:
        if r < goal[0] and c < goal[1]:
            if random.random() < 0.5: r += 1
            else: c += 1
        elif r < goal[0]: r += 1
        elif c < goal[1]: c += 1
        grid[r][c] = 1
    for _ in range(7):
        rh, rw = random.randint(3, 5), random.randint(3, 5)
        rr, rc = random.randint(1, rows - rh - 1), random.randint(1, cols - rw - 1)
        for i in range(rr, rr + rh):
            for j in range(rc, rc + rw): grid[i][j] = 1
    for _ in range(25):
        rr, rc = random.randint(1, rows - 2), random.randint(1, cols - 2)
        grid[rr][rc] = 1
    return grid

# ==========================================
# 2. ALGORITHMS
# ==========================================
def bfs(grid, start, goal):
    queue = deque([start]); visited = {start}; parent = {start: None}; order = []
    while queue:
        cell = queue.popleft(); order.append(cell)
        if cell == goal: return reconstruct_path(parent, start, goal), order
        for nb in get_neighbors(grid, *cell):
            if nb not in visited:
                visited.add(nb); parent[nb] = cell; queue.append(nb)
    return [], order

def dfs(grid, start, goal):
    stack = [start]; visited = {start}; parent = {start: None}; order = []
    while stack:
        cell = stack.pop(); order.append(cell)
        if cell == goal: return reconstruct_path(parent, start, goal), order
        for nb in get_neighbors(grid, *cell):
            if nb not in visited:
                visited.add(nb); parent[nb] = cell; stack.append(nb)
    return [], order

def astar(grid, start, goal):
    h = lambda c: manhattan(c, goal); open_list = [(h(start), 0, start)]
    g_scores = {start: 0}; parent = {start: None}; order = []; closed = set()
    while open_list:
        f, g, cell = heapq.heappop(open_list)
        if cell in closed: continue
        closed.add(cell); order.append(cell)
        if cell == goal: return reconstruct_path(parent, start, goal), order
        for nb in get_neighbors(grid, *cell):
            new_g = g + 1
            if nb not in g_scores or new_g < g_scores[nb]:
                g_scores[nb] = new_g; parent[nb] = cell
                heapq.heappush(open_list, (new_g + h(nb), new_g, nb))
    return [], order

# ==========================================
# 3. UNIFIED UI DASHBOARD (Plotting All in One Figure)
# ==========================================
def render_algorithm_panel():
    start, goal = (1, 1), (18, 22)
    grid = generate_dungeon(20, 24, start, goal)
    
    # Run algos
    bfs_path, bfs_vis = bfs(grid, start, goal)
    dfs_path, dfs_vis = dfs(grid, start, goal)
    ast_path, ast_vis = astar(grid, start, goal)
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor="#111111")
    fig.suptitle("AI Quest Game — Algorithm Visualization", color="gold", fontsize=20, fontweight="bold")
    
    algos = [
        ("BFS", bfs_path, bfs_vis, axes[0,0], "#3498DB"),
        ("DFS", dfs_path, dfs_vis, axes[0,1], "#9B59B6"),
        ("A*", ast_path, ast_vis, axes[0,2], "#2ECC71"),
    ]
    
    for title, path, vis, ax, color in algos:
        # Base grid plotting
        display_data = np.zeros((20, 24, 3))
        for r in range(20):
            for c in range(24):
                display_data[r, c] = [0.2, 0.2, 0.2] if grid[r][c] == 0 else [0.8, 0.8, 0.8]
        for (r,c) in vis: display_data[r, c] = [int(color[1:3],16)/255/1.5, int(color[3:5],16)/255/1.5, int(color[5:7],16)/255/1.5]
        for (px, py) in path: display_data[px, py] = [int(color[1:3],16)/255, int(color[3:5],16)/255, int(color[5:7],16)/255]
        display_data[start[0], start[1]] = [0.1, 0.8, 0.8] # Start
        display_data[goal[0], goal[1]] = [0.9, 0.7, 0.1] # Goal
        
        ax.imshow(display_data)
        ax.set_title(title, color="white")
        ax.axis("off")
        
    axes[1,0].set_title("Hill Climbing (Not Executed)", color="white"); axes[1,0].axis("off")
    axes[1,1].set_title("Minimax (Not Executed)", color="white"); axes[1,1].axis("off")
    axes[1,2].set_title("K-Means (Not Executed)", color="white"); axes[1,2].axis("off")

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. IPYWIDGETS CONTROLS
# ==========================================
out = widgets.Output()

btn_run = widgets.Button(description="Refresh Dashboard", button_style="success")

def on_run_clicked(b):
    with out:
        clear_output(wait=True)
        render_algorithm_panel()

btn_run.on_click(on_run_clicked)

display(widgets.VBox([
    widgets.HTML("<h2 style='color: gold;'>AI Dashboard Controls</h2>"),
    btn_run,
    out
]))

# Trigger first draw automatically
on_run_clicked(None)